In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from scripts.model import DonutConfig, DonutModel
from scripts.tokenizer import get_tokenizer

W0930 19:07:11.874000 20842 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 19:07:11.902000 20842 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
tokenizer = get_tokenizer()
config = DonutConfig(
    input_size=(1280, 960),
    window_size=10,
    encoder_layer=(1, 1, 1, 1),
    max_position_embeddings=768,
    decoder_layer=1,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)
donut_model = DonutModel(config).to(device)

state_dict = torch.load(
    "donut_model.pth",
    map_location=device,
)

donut_model.load_state_dict(state_dict)
donut_model.eval()
pass

In [4]:
# fix timm buffers
for m in donut_model.modules():
    if hasattr(m, "init_non_persistent_buffers"):
        m.init_non_persistent_buffers()

In [5]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

In [7]:
result = donut_model.predict(sample["image"], tokenizer)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [17]:
result

{'predictions': [{'bill_to': {'name': 'PT Maju Teknologi',
    'address': 'Gg. Tebet Barat Dalam No. 40',
    'city': 'Bontang, Indonesia',
    'phone': '+62 823-9935-2424',
    'email': 'finance@majuteknologi.com'},
   'invoice_date': '1 February 2025',
   'invoice_no': 'INV-2025-30496',
   'from_to': {'name': 'PT Solusi Nusantara',
    'address': 'Gang Antapani Lama No. 34',
    'city': 'Meulaboh, Indonesia',
    'phone': '+62 885-7912-1520',
    'email': 'billing@solusinusantara.com'},
   'items': [{'description': 'DevOps Services',
     'qty': '1',
     'price': '3800000',
     'total': '3800000'},
    {'description': 'UI/UX Design',
     'qty': '9',
     'price': '2900000',
     'total': '26100000'},
    {'description': 'UI/UX Design',
     'qty': '8',
     'price': '4000000',
     'total': '32000000'},
    {'description': 'Cloud Infrastructure',
     'qty': '1',
     'price': '1250000',
     'total': '1250000'},
    {'description': 'AI Development',
     'qty': '7',
     'price':

### Set Folder

In [1]:
repo_dir = "./invoice-doc-to-text"

In [6]:
config.auto_map = {
    "AutoConfig": "donut.DonutConfig",
    "AutoModel": "donut.DonutModel",
}
donut_model.config = config

import os
os.makedirs(repo_dir, exist_ok=True)

donut_model.save_pretrained(repo_dir)
tokenizer.save_pretrained(repo_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./invoice-doc-to-text/tokenizer_config.json',
 './invoice-doc-to-text/tokenizer.json')

### Upload to HF

In [2]:
from huggingface_hub import login, upload_folder
from datasets import load_dataset
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi
load_dotenv(override=True)

login(token=os.environ["HF_TOKEN"])

repo_id = "ridwanFatur98/invoice-doc-to-text"

api = HfApi()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
# api.create_repo(
#     repo_id=repo_id,
#     repo_type="model",
#     exist_ok=True,
# )

# api.upload_folder(
#     folder_path=repo_dir,
#     repo_id=repo_id,
#     repo_type="model",
# )

In [3]:
# api.upload_file(
#     path_or_fileobj=f"{repo_dir}/donut.py",
#     path_in_repo="donut.py",
#     repo_id=repo_id,
#     repo_type="model",
# )

In [4]:
# api.upload_file(
#     path_or_fileobj=f"{repo_dir}/image_process.py",
#     path_in_repo="image_process.py",
#     repo_id=repo_id,
#     repo_type="model",
# )

In [5]:
# api.upload_file(
#     path_or_fileobj=f"{repo_dir}/inference.py",
#     path_in_repo="inference.py",
#     repo_id=repo_id,
#     repo_type="model",
# )

In [7]:
api.upload_file(
    path_or_fileobj=f"{repo_dir}/README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/ridwanFatur98/invoice-doc-to-text/commit/e7f6a3949e681ac71c7a019f6b4ff9731139bdc4', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='e7f6a3949e681ac71c7a019f6b4ff9731139bdc4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ridwanFatur98/invoice-doc-to-text', endpoint='https://huggingface.co', repo_type='model', repo_id='ridwanFatur98/invoice-doc-to-text'), pr_revision=None, pr_num=None)

In [46]:
# from huggingface_hub import HfApi

# repo_id = "ridwanFatur98/invoice-doc-to-text"

# api = HfApi()

# api.delete_repo(
#     repo_id=repo_id,
#     repo_type="model",
# )

# print("Repository deleted.")